# Tarea — Análisis de Logs y Detección de Cuellos de Botella con PySpark

### Objetivo
Procesar logs simulados de microservicios y detectar anomalías de rendimiento usando PySpark.

## Instrucciones
1. Genera archivos de log simulados (CSV) con datos de timestamp, service, endpoint, response_time_ms, status_code.
2. Carga los archivos con Spark.
3. Limpia los datos (nulos, tipos incorrectos).
4. Calcula KPIs: tiempo promedio, desviación estándar y porcentaje de errores.
5. Detecta outliers en response_time_ms mediante z-score.
6. Guarda los resultados procesados en formato Parquet.

In [ ]:
# Crear sesión Spark
# TODO: inicializa SparkSession

from pyspark.sql import SparkSession

spark = (SparkSession.builder
    .appName("AnalisisLogsMicroservicios")
    .master("local[*]")  # Usar todos los cores locales
    .getOrCreate())

print(f"SparkSession iniciada. Versión: {spark.version}")


Py4JJavaError: An error occurred while calling None.org.apache.spark.api.java.JavaSparkContext.
: java.lang.UnsupportedOperationException: getSubject is supported only if a security manager is allowed
	at java.base/javax.security.auth.Subject.getSubject(Subject.java:347)
	at org.apache.hadoop.security.UserGroupInformation.getCurrentUser(UserGroupInformation.java:588)
	at org.apache.spark.util.Utils$.$anonfun$getCurrentUserName$1(Utils.scala:2446)
	at scala.Option.getOrElse(Option.scala:201)
	at org.apache.spark.util.Utils$.getCurrentUserName(Utils.scala:2446)
	at org.apache.spark.SparkContext.<init>(SparkContext.scala:339)
	at org.apache.spark.api.java.JavaSparkContext.<init>(JavaSparkContext.scala:59)
	at java.base/jdk.internal.reflect.DirectConstructorHandleAccessor.newInstance(DirectConstructorHandleAccessor.java:62)
	at java.base/java.lang.reflect.Constructor.newInstanceWithCaller(Constructor.java:501)
	at java.base/java.lang.reflect.Constructor.newInstance(Constructor.ja

In [ ]:
# Generar logs simulados y guardarlos en CSV
# TODO: genera un dataset sintético con pandas o PySpark

import pandas as pd
import numpy as np
import random
from datetime import datetime, timedelta
import os

# --- Configuración de la simulación ---
NUM_RECORDS = 10000
LOG_DIR = "logs_data"
LOG_FILE = os.path.join(LOG_DIR, "microservice_logs.csv")

# Crear directorio si no existe
if not os.path.exists(LOG_DIR):
    os.makedirs(LOG_DIR)

# --- Definir servicios y endpoints ---
services_endpoints = {
    'auth-service': ['/login', '/register', '/validate_token'],
    'payment-service': ['/process_payment', '/payment_history', '/refund'],
    'user-service': ['/get_profile', '/update_profile', '/list_users']
}

# Distribución de códigos de estado (más éxitos que errores)
status_codes_dist = {
    200: 0.7, 201: 0.1, 400: 0.05, 401: 0.05, 404: 0.05, 500: 0.05
}
status_codes = list(status_codes_dist.keys())
status_probs = list(status_codes_dist.values())

data = []
start_time = datetime.now() - timedelta(days=1)

print(f"Generando {NUM_RECORDS} registros simulados...")

for i in range(NUM_RECORDS):
    service = random.choice(list(services_endpoints.keys()))
    endpoint = random.choice(services_endpoints[service])
    timestamp = (start_time + timedelta(seconds=i)).isoformat()
    
    # --- Simular tiempos de respuesta (con anomalías) ---
    base_time = 50
    if service == 'payment-service':
        base_time = 150 # 'payments' es más lento por diseño
    
    # Tiempos normales (distribución normal)
    response_time = np.random.normal(loc=base_time, scale=40)
    
    # Introducir outliers (2% de las veces)
    if random.random() < 0.02: 
        response_time *= random.uniform(8, 15) # Aumentar drásticamente
    
    # Introducir nulos (1% de las veces)
    if random.random() < 0.01:
        response_time = np.nan
        
    # Asegurar que no sea negativo
    if response_time < 0:
        response_time = base_time / 3

    status = np.random.choice(status_codes, p=status_probs)
    
    # Introducir nulos en otras columnas (1% de las veces)
    if random.random() < 0.01:
        service = None

    data.append([timestamp, service, endpoint, response_time, status])

# --- Crear DataFrame de Pandas y guardar en CSV ---
df_pandas = pd.DataFrame(data, columns=['timestamp', 'service', 'endpoint', 'response_time_ms', 'status_code'])
df_pandas.to_csv(LOG_FILE, index=False, header=True)

print(f"Logs generados y guardados en '{LOG_FILE}'")
print(df_pandas.head())

In [ ]:
# Cargar logs con Spark
# TODO: usa spark.read.csv con header=True



In [ ]:
# Limpiar y transformar datos
# TODO: convertir columnas a tipo adecuado y filtrar valores inválidos



In [ ]:
# Calcular KPIs
# TODO: agrupa por service y endpoint y calcula métricas



In [ ]:
# Detectar anomalías por z-score
# TODO: usa ventanas y PySpark.sql.functions

